# LA 2026-06-15 detailed files: initial inspection only
Read listings.csv.gz, calendar.csv.gz, reviews.csv.gz and neighbourhoods.csv. Source files and raw DataFrames are not cleaned or merged.
Grain: Listings = one listing per snapshot; Calendar = one listing-date; Reviews = one review; Neighborhoods = one neighborhood reference record.


In [ ]:
import pandas as pd
from pathlib import Path

# Works when the notebook is started from project/ or the workspace root.
data_folder = Path("group data/2026-06-15")
if not data_folder.is_dir():
    data_folder = Path("project/group data/2026-06-15")
if not data_folder.is_dir():
    raise FileNotFoundError("Cannot find the 2026-06-15 data folder.")

print("Data folder:", data_folder.resolve())
listings = pd.read_csv(data_folder / "listings.csv.gz", low_memory=False)
calendar = pd.read_csv(data_folder / "calendar.csv.gz", low_memory=False)
reviews = pd.read_csv(data_folder / "reviews.csv.gz", low_memory=False)
neighborhoods = pd.read_csv(data_folder / "neighbourhoods.csv", low_memory=False)

tables = {"Listings": listings, "Calendar": calendar,
          "Reviews": reviews, "Neighborhoods": neighborhoods}
for name, table in tables.items():
    print(name, "shape:", table.shape)
    display(table.head())


All missing-value and duplicate checks below use the full tables. Duplicate counts show extra occurrences after the first row.

In [ ]:
# Inspect all rows, not just head().
for name, table in tables.items():
    print("\n" + name)
    print("Columns:", table.columns.tolist())
    print("Data types:")
    print(table.dtypes.to_string())
    print("Missing values:")
    print(table.isna().sum().to_string())
    print("Exact duplicate rows:", table.duplicated().sum())


In [ ]:
print("Listings: missing id:", listings["id"].isna().sum())
print("Listings: repeated id:", listings["id"].duplicated().sum())
print("Calendar: missing key values:")
print(calendar[["listing_id", "date"]].isna().sum())
print("Calendar: repeated listing-date:",
      calendar.duplicated(subset=["listing_id", "date"]).sum())

if "id" in reviews.columns:
    print("Reviews: missing review id:", reviews["id"].isna().sum())
    print("Reviews: repeated review id:", reviews["id"].duplicated().sum())
else:
    print("Reviews: no review id column; review key uniqueness cannot be checked.")
    print("Repeated listing-date rows (not proof of duplicate reviews):",
          reviews.duplicated(subset=["listing_id", "date"]).sum())

print("Neighborhoods: missing neighbourhood:",
      neighborhoods["neighbourhood"].isna().sum())
print("Neighborhoods: repeated neighbourhood:",
      neighborhoods["neighbourhood"].duplicated().sum())


Check field availability before planning cleaning. Unusual prices or stay limits are investigation candidates, not confirmed errors. Text dates have not been converted.

In [ ]:
project_fields = [
    "id", "price", "neighbourhood_cleansed", "room_type", "bedrooms",
    "property_type", "accommodates", "minimum_nights", "availability_90",
    "number_of_reviews_ltm", "review_scores_value", "scrape_id", "last_scraped"
]
field_check = pd.DataFrame({"field": project_fields})
field_check["present"] = field_check["field"].isin(listings.columns)
field_check["missing_count"] = [
    listings[field].isna().sum() if field in listings.columns else None
    for field in project_fields
]
display(field_check)

print("Listings numerical ranges:")
print(listings[["bedrooms", "accommodates", "minimum_nights", "number_of_reviews_ltm",
                "availability_90", "review_scores_value", "latitude", "longitude"]].agg(["min", "max"]).to_string())
print("Room types:")
print(listings["room_type"].value_counts(dropna=False))
# Temporary inspection only: do not change the raw price column.
price_check = pd.to_numeric(
    listings["price"].astype("string").str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False), errors="coerce"
)
print("Price range:", price_check.min(), price_check.max())
print("Nonmissing prices that could not be interpreted:",
      (listings["price"].notna() & price_check.isna()).sum())
print("Zero or negative observed prices:", (price_check <= 0).sum())
print("Zero or negative minimum nights:", (listings["minimum_nights"] <= 0).sum())
print("Calendar available values:")
print(calendar["available"].value_counts(dropna=False))
print("Calendar date range:", calendar["date"].min(), calendar["date"].max())
calendar_counts = calendar.groupby("listing_id").size()
print("Calendar rows per listing:")
print(calendar_counts.value_counts().sort_index())
print("Reviews date range:", reviews["date"].min(), reviews["date"].max())
print("Host ID types (check against original text before using them as keys):")
print(listings[["host_id", "host_profile_id"]].dtypes)

print("Scrape IDs:")
print(listings["scrape_id"].value_counts(dropna=False).to_string())
print("Last scraped dates:")
print(listings["last_scraped"].value_counts(dropna=False).to_string())
print("Missing value scores by total review count:")
print(listings.loc[listings["review_scores_value"].isna(),
                   "number_of_reviews"].value_counts(dropna=False).to_string())


## 1. Baseline evidence
Record the raw tables before creating cleaned copies. Missing counts cover every column. Key duplicates count extra occurrences after the first.


In [ ]:
keys = {"Listings": ["id"], "Calendar": ["listing_id", "date"],
        "Reviews": ["id"], "Neighborhoods": ["neighbourhood"]}
baseline_rows = []
baseline_missing_rows = []
for name, table in tables.items():
    baseline_rows.append({
        "table": name, "rows": len(table), "columns": len(table.columns),
        "missing_key_rows": table[keys[name]].isna().any(axis=1).sum(),
        "duplicate_keys": table.duplicated(subset=keys[name]).sum()
    })
    for column in table.columns:
        baseline_missing_rows.append({
            "table": name, "column": column,
            "missing_count": table[column].isna().sum()
        })
baseline = pd.DataFrame(baseline_rows)
baseline_missing = pd.DataFrame(baseline_missing_rows)
display(baseline)
display(baseline_missing)


## 2. Cleaned copies
Keep raw DataFrames unchanged. Only approved derived columns will be added to independent copies.


In [ ]:
listings_clean = listings.copy()
calendar_clean = calendar.copy()
reviews_clean = reviews.copy()
neighborhoods_clean = neighborhoods.copy()
clean_tables = {"Listings": listings_clean, "Calendar": calendar_clean,
                "Reviews": reviews_clean, "Neighborhoods": neighborhoods_clean}


## 3. Price conversion
The raw price is currency text. Add price_numeric to support statistics; retain missing values and all unusual prices. A conversion failure is a nonmissing original value that becomes missing.


In [ ]:
listings_clean["price_numeric"] = pd.to_numeric(
    listings_clean["price"].astype("string")
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)
price_failures = (listings["price"].notna()
                  & listings_clean["price_numeric"].isna()).sum()
price_evidence = pd.DataFrame([{
    "conversion_failures": price_failures,
    "missing_before": listings["price"].isna().sum(),
    "missing_after": listings_clean["price_numeric"].isna().sum(),
    "min": listings_clean["price_numeric"].min(),
    "max": listings_clean["price_numeric"].max(),
    "zero_or_negative": (listings_clean["price_numeric"] <= 0).sum()
}])
display(price_evidence)
print("Original and converted examples:")
display(listings_clean[["id", "price", "price_numeric"]].head(5))
print("Missing-price examples (retained):")
display(listings_clean.loc[listings_clean["price"].isna(),
                           ["id", "price", "price_numeric"]].head(3))
assert price_failures == 0
assert listings["price"].isna().equals(listings_clean["price_numeric"].isna())


## 4. Date conversion
Add datetime columns using YYYY-MM-DD. Preserve each original date, including the multiple last_scraped dates. Verify every row by converting parsed dates back to text.


In [ ]:
date_specs = [
    ("Listings", listings_clean, "last_scraped", "last_scraped_dt"),
    ("Calendar", calendar_clean, "date", "date_dt"),
    ("Reviews", reviews_clean, "date", "date_dt")
]
date_rows = []
for name, table, original, parsed in date_specs:
    table[parsed] = pd.to_datetime(
        table[original], format="%Y-%m-%d", errors="coerce"
    )
    failures = (table[original].notna() & table[parsed].isna()).sum()
    parsed_text = table[parsed].dt.strftime("%Y-%m-%d")
    matching = (table[original].eq(parsed_text)
                | (table[original].isna() & table[parsed].isna()))
    date_rows.append({
        "table": name, "parsing_failures": failures,
        "mismatched_rows": (~matching).sum(),
        "missing_before": table[original].isna().sum(),
        "missing_after": table[parsed].isna().sum(),
        "min_date": table[parsed].min(), "max_date": table[parsed].max()
    })
    assert failures == 0
    assert matching.all()
    del parsed_text, matching
date_evidence = pd.DataFrame(date_rows)
display(date_evidence)
print("Listings last_scraped frequencies:")
display(listings_clean["last_scraped_dt"].value_counts(dropna=False).sort_index())


## 5. Important missing values
Document missingness without filling values or removing records.


In [ ]:
missing_specs = [
    ("Listings", "price"), ("Listings", "bedrooms"),
    ("Listings", "minimum_nights"), ("Listings", "review_scores_value"),
    ("Reviews", "reviewer_name"), ("Reviews", "comments")
]
missing_rows = []
for name, column in missing_specs:
    before = tables[name][column].isna().sum()
    after = clean_tables[name][column].isna().sum()
    missing_rows.append({"table": name, "column": column,
                         "missing_before": before, "missing_after": after})
    assert before == after
missing_evidence = pd.DataFrame(missing_rows)
display(missing_evidence)


## 6. Key validation
Validate the cleaned copies without deduplicating. Calendar uses the original date in its composite key.


In [ ]:
key_rows = []
for name, table in clean_tables.items():
    key_rows.append({
        "table": name, "key": " + ".join(keys[name]),
        "missing_key_rows": table[keys[name]].isna().any(axis=1).sum(),
        "duplicate_keys": table.duplicated(subset=keys[name]).sum()
    })
key_evidence = pd.DataFrame(key_rows)
display(key_evidence)
assert (key_evidence["missing_key_rows"] == 0).all()
assert (key_evidence["duplicate_keys"] == 0).all()


## 7. Range validation
Report ranges without capping or deleting values. Checks flag values inconsistent with field meanings: nonpositive prices/stay limits/capacity, negative bedrooms, scores outside 0?5, and availability outside 0?90. Flags require investigation. High prices, zero bedrooms, long minimum stays, and zero scores are not automatically errors.


In [ ]:
range_columns = ["price_numeric", "bedrooms", "minimum_nights",
                 "review_scores_value", "availability_90", "accommodates"]
range_checks = {
    "price_numeric": listings_clean["price_numeric"] <= 0,
    "bedrooms": listings_clean["bedrooms"] < 0,
    "minimum_nights": listings_clean["minimum_nights"] <= 0,
    "review_scores_value": ((listings_clean["review_scores_value"] < 0)
                            | (listings_clean["review_scores_value"] > 5)),
    "availability_90": ((listings_clean["availability_90"] < 0)
                        | (listings_clean["availability_90"] > 90)),
    "accommodates": listings_clean["accommodates"] <= 0
}
range_rows = []
for column in range_columns:
    range_rows.append({
        "column": column, "min": listings_clean[column].min(),
        "max": listings_clean[column].max(),
        "outside_meaningful_bounds": range_checks[column].sum()
    })
    if range_checks[column].any():
        print("Investigate:", column)
        display(listings_clean.loc[range_checks[column], ["id", column]].head(5))
range_evidence = pd.DataFrame(range_rows)
display(range_evidence)


## 8. Calendar coverage
Use isin to check identifiers without merging or deleting unmatched observations. Different coverage is an unresolved limitation.


In [ ]:
calendar_unmatched = ~calendar["listing_id"].isin(listings["id"])
unmatched_calendar_ids = calendar.loc[
    calendar_unmatched, "listing_id"
].drop_duplicates()
listings_without_calendar = listings.loc[
    ~listings["id"].isin(calendar["listing_id"]), "id"
]
coverage_evidence = pd.DataFrame([{
    "distinct_unmatched_calendar_ids": len(unmatched_calendar_ids),
    "calendar_rows_for_unmatched_ids": calendar_unmatched.sum(),
    "listings_without_calendar": len(listings_without_calendar)
}])
display(coverage_evidence)
print("Sample Calendar IDs not in Listings:")
display(unmatched_calendar_ids.head(10))
print("Sample Listings IDs not in Calendar:")
display(listings_without_calendar.head(10))


## 9. Reconciliation
Check every original column and row against the raw DataFrames. Only price_numeric and the three approved datetime columns may be added.


In [ ]:
approved_columns = {
    "Listings": ["price_numeric", "last_scraped_dt"],
    "Calendar": ["date_dt"], "Reviews": ["date_dt"], "Neighborhoods": []
}
reconciliation_rows = []
for name, raw in tables.items():
    cleaned = clean_tables[name]
    added = [column for column in cleaned.columns if column not in raw.columns]
    originals_unchanged = cleaned[raw.columns].equals(raw)
    reconciliation_rows.append({
        "table": name, "raw_rows": len(raw), "clean_rows": len(cleaned),
        "row_change": len(cleaned) - len(raw),
        "original_columns_unchanged": originals_unchanged,
        "added_columns": ", ".join(added),
        "only_approved_columns_added": added == approved_columns[name]
    })
    assert len(cleaned) == len(raw)
    assert originals_unchanged
    assert added == approved_columns[name]
reconciliation = pd.DataFrame(reconciliation_rows)
display(reconciliation)


## Stop after the approved first batch
No missing values were filled, no rows were dropped, no unusual values were removed or capped, and no tables were merged. Evidence tables above support the decision log. Missingness policy, special records, host_profile_id precision, Calendar window differences, and unmatched listing coverage remain unresolved.


## 10. Investigation only: bedrooms
Diagnostic counts do not change analysis tables. Use 10+ bedrooms only to select examples for review, not as an error threshold.


In [ ]:
example_columns = ["id", "name", "room_type", "property_type", "bedrooms",
                   "accommodates", "minimum_nights", "price", "price_numeric"]
for column in ["room_type", "property_type"]:
    counts = listings_clean.groupby(column, dropna=False)["bedrooms"].agg(["size", "count"])
    counts["missing"] = counts["size"] - counts["count"]
    counts["missing_pct"] = 100 * counts["missing"] / counts["size"]
    print("Bedroom missingness by", column)
    display(counts.sort_values("missing", ascending=False))
print("Missing bedrooms examples:")
display(listings_clean.loc[listings_clean["bedrooms"].isna(), example_columns].head(8))
print("Zero bedrooms:", listings_clean["bedrooms"].eq(0).sum())
display(listings_clean.loc[listings_clean["bedrooms"].eq(0), example_columns].head(8))
print("10+ bedrooms:", listings_clean["bedrooms"].ge(10).sum())
display(listings_clean.loc[listings_clean["bedrooms"].ge(10), example_columns]
        .sort_values("bedrooms", ascending=False).head(10))


## 11. Investigation only: minimum stay
Retain missing and high values. Examine distributions and listing descriptions before choosing comparison scope.


In [ ]:
print("All missing minimum_nights records:")
display(listings_clean.loc[listings_clean["minimum_nights"].isna(), example_columns])
print("Minimum nights quantiles:")
display(listings_clean["minimum_nights"].quantile([0, .25, .5, .75, .9, .95, .99, 1]))
print("Largest minimum stays:")
display(listings_clean.sort_values("minimum_nights", ascending=False)[example_columns].head(12))
print("Stay counts at investigation cutoffs:")
for cutoff in [28, 30, 90, 365]:
    print("Minimum nights >=", cutoff, ":", listings_clean["minimum_nights"].ge(cutoff).sum())
print("365+ nights by room/property type (investigation subset, not exclusions):")
display(listings_clean.loc[listings_clean["minimum_nights"].ge(365)]
        .groupby(["room_type", "property_type"]).size().sort_values(ascending=False))


## 12. Investigation only: value scores
Missing is not zero. Check all reviewed-but-unscored and zero-score records.


In [ ]:
print("Missing score by total review count:")
display(listings_clean.loc[listings_clean["review_scores_value"].isna(),
                          "number_of_reviews"].value_counts().sort_index())
score_columns = ["id", "name", "room_type", "property_type", "number_of_reviews",
                 "number_of_reviews_ltm", "first_review", "last_review",
                 "review_scores_rating", "review_scores_value"]
print("Reviewed listings with missing value scores:")
display(listings_clean.loc[
    listings_clean["review_scores_value"].isna()
    & listings_clean["number_of_reviews"].gt(0), score_columns])
print("Zero value scores:", listings_clean["review_scores_value"].eq(0).sum())
display(listings_clean.loc[listings_clean["review_scores_value"].eq(0), score_columns])
print("Nonmissing scores outside 0?5:",
      ((listings_clean["review_scores_value"] < 0)
       | (listings_clean["review_scores_value"] > 5)).sum())


## 13. Investigation only: price extremes
Inspect quote dates, totals and per-night prices; quote consistency supports the extraction but does not prove market comparability.


In [ ]:
quote_columns = ["price_quote_checkin_date", "price_quote_checkout_date",
                 "price_quote_total_price", "price_quote_price_per_night"]
low_prices = listings_clean.nsmallest(8, "price_numeric")
high_prices = listings_clean.nlargest(8, "price_numeric")
for label, examples in [("Lowest prices", low_prices), ("Highest prices", high_prices)]:
    print(label)
    display(examples[example_columns + quote_columns])
    for row_index, row in examples.head(3).iterrows():
        print("Listing", row["id"], "description:", str(row["description"])[:700])
        print("Raw quote:", str(row["price_quote_raw"])[:1600])


## 14. Investigation only: Calendar coverage and windows
Use diagnostic grouping to describe source coverage; no listing-level analysis table or merge is produced. Compare observed counts with inclusive date spans. Extra dates are defined relative to each listing's first observed date, without assuming that this is the intended window.


In [ ]:
calendar_window_evidence = calendar_clean.groupby("listing_id")["date_dt"].agg(
    ["size", "nunique", "min", "max"])
calendar_window_evidence["inclusive_span_days"] = (
    calendar_window_evidence["max"] - calendar_window_evidence["min"]
).dt.days + 1
calendar_window_evidence["consecutive"] = (
    calendar_window_evidence["nunique"]
    == calendar_window_evidence["inclusive_span_days"])
print("All Calendar window patterns:")
display(calendar_window_evidence.groupby(
    ["size", "inclusive_span_days", "consecutive"]).size())
unmatched_windows = calendar_window_evidence.loc[
    ~calendar_window_evidence.index.isin(listings["id"])]
print("Unmatched IDs:", len(unmatched_windows),
      "Calendar rows:", unmatched_windows["size"].sum())
print("Unmatched window counts:")
display(unmatched_windows.groupby(["size", "min", "max", "consecutive"]).size())
print("Unmatched sample windows:")
display(unmatched_windows.head(10))
print("Unmatched availability values:")
display(calendar.loc[calendar_unmatched, "available"].value_counts())
print("Unmatched minimum_nights range:",
      calendar.loc[calendar_unmatched, "minimum_nights"].min(),
      calendar.loc[calendar_unmatched, "minimum_nights"].max())

extended_windows = calendar_window_evidence.loc[
    calendar_window_evidence["size"].isin([366, 367])].copy()
extended_windows["first_date_after_365"] = (
    extended_windows["min"] + pd.Timedelta(days=365))
print("All 366/367 windows:")
display(extended_windows)
print("Listings for these IDs, where available:")
display(listings_clean.loc[
    listings_clean["id"].isin(extended_windows.index),
    ["id", "name", "last_scraped", "calendar_last_scraped", "room_type"]])
print("Every extra date after the first 365 days:")
for listing_id, window in extended_windows.iterrows():
    extra = calendar_clean.loc[
        calendar_clean["listing_id"].eq(listing_id)
        & calendar_clean["date_dt"].ge(window["first_date_after_365"]),
        ["listing_id", "date", "available"]]
    display(extra)


## 15. Confirm investigation did not change data
Repeat row and original-column reconciliation. No proposed treatment has been implemented.


In [ ]:
for name, raw in tables.items():
    cleaned = clean_tables[name]
    assert len(cleaned) == len(raw)
    assert cleaned[raw.columns].equals(raw)
    assert [c for c in cleaned.columns if c not in raw.columns] == approved_columns[name]
print("All raw columns and row counts unchanged; only first-batch columns exist.")


## 16. Final decision log and existing text checks
These are the approved rules from the script. No new cleaning rule or investigation is added. Original category labels remain unchanged.


In [ ]:
decision_log = pd.DataFrame([
    {"rule": "Price numeric conversion", "reason": "Currency text cannot support numerical price comparisons", "check": "0 failures; missing mask preserved; original price unchanged", "type": "Type conversion", "status": "Implemented"},
    {"rule": "Explicit date parsing", "reason": "Support date windows while retaining actual scrape dates", "check": "0 failures and 0 round-trip mismatches", "type": "Type conversion", "status": "Implemented"},
    {"rule": "Retain missing values", "reason": "Unknown bedroom/price/score is not zero; no justified imputation", "check": "Missing counts before and after identical", "type": "Missing-data decision", "status": "Implemented: keep"},
    {"rule": "Retain unique source records", "reason": "All four table keys are unique", "check": "Exact duplicates and repeated keys checked separately", "type": "Duplicate decision", "status": "Implemented: keep"},
    {"rule": "Eligibility indicators only", "reason": "State metric denominators and unknown bedroom coverage without deleting records", "check": "Flags equal the stated nonmissing conditions on every row", "type": "Analysis scope", "status": "Apply below"},
    {"rule": "Monthly quote investigation flag", "reason": "Extreme low prices include ambiguous Average monthly price quotes", "check": "Flag matches the literal quote text; raw price remains unchanged", "type": "Limitation flag", "status": "Apply below; interpretation unresolved"},
    {"rule": "Retain extremes and zero scores", "reason": "Some extremes are plausible; correction lacks source evidence", "check": "Raw columns identical; examples retained", "type": "Validation / limitation", "status": "Keep, investigate"},
    {"rule": "Keep unmatched Calendar and extended windows", "reason": "181 unmatched IDs and 14 extended windows are not proven errors", "check": "No row-count change; coverage/window evidence retained", "type": "Limitation", "status": "Keep"},
    {"rule": "Do not repair host_profile_id from float", "reason": "Possible integer precision loss needs original text verification", "check": "Not used as a merge or analytical key", "type": "Limitation", "status": "Unresolved"}
])
display(decision_log)
text_rows = []
for name, column in [
    ("Listings", "neighbourhood_cleansed"), ("Listings", "room_type"),
    ("Listings", "property_type"), ("Neighborhoods", "neighbourhood")
]:
    values = clean_tables[name][column].astype("string")
    changed = values.notna() & values.ne(values.str.strip())
    blank = values.notna() & values.str.strip().eq("")
    text_rows.append({"table": name, "column": column,
                      "surrounding_whitespace": changed.sum(),
                      "blank_nonmissing": blank.sum()})
    if changed.any():
        print("Investigate whitespace; original values retained:", name, column)
        display(clean_tables[name].loc[changed, [column]].head())
text_evidence = pd.DataFrame(text_rows)
display(text_evidence)
print("Categorical examples, unchanged:")
display(listings_clean[["neighbourhood_cleansed", "room_type", "property_type"]].head())


## 17. Final analysis eligibility flags
Flags describe required-field completeness, not guaranteed price comparability. Monthly quote text is flagged for review only; no rows or prices are removed.


In [ ]:
listings_clean["has_observed_price"] = listings_clean["price_numeric"].notna()
listings_clean["has_bedroom_segment"] = (
    listings_clean["neighbourhood_cleansed"].notna()
    & listings_clean["room_type"].notna()
    & listings_clean["bedrooms"].notna()
)
listings_clean["eligible_price_bedroom_segment"] = (
    listings_clean["has_observed_price"] & listings_clean["has_bedroom_segment"]
)
listings_clean["monthly_quote_review_flag"] = (
    listings_clean["price_quote_raw"].astype("string")
    .str.contains("Average monthly price", regex=False, na=False)
)
approved_columns["Listings"] = [
    "price_numeric", "last_scraped_dt", "has_observed_price",
    "has_bedroom_segment", "eligible_price_bedroom_segment", "monthly_quote_review_flag"
]
assert listings_clean["has_observed_price"].equals(listings_clean["price_numeric"].notna())
assert listings_clean["has_bedroom_segment"].equals(
    listings_clean[["neighbourhood_cleansed", "room_type", "bedrooms"]].notna().all(axis=1))
assert listings_clean["eligible_price_bedroom_segment"].equals(
    listings_clean["has_observed_price"] & listings_clean["has_bedroom_segment"])
denominator_evidence = pd.DataFrame([
    {"metric": "Observed price mean/median", "valid_records": listings_clean["price_numeric"].count(),
     "missing_records": listings_clean["price_numeric"].isna().sum(),
     "value": listings_clean["price_numeric"].median()},
    {"metric": "Value-score mean including retained zero", "valid_records": listings_clean["review_scores_value"].count(),
     "missing_records": listings_clean["review_scores_value"].isna().sum(),
     "value": listings_clean["review_scores_value"].mean()},
    {"metric": "Price + bedroom segment complete records", "valid_records": listings_clean["eligible_price_bedroom_segment"].sum(),
     "missing_records": (~listings_clean["eligible_price_bedroom_segment"]).sum(), "value": None}
])
display(denominator_evidence)
print("Monthly-quote review flags:", listings_clean["monthly_quote_review_flag"].sum())
print("Means and medians skip missing values; missing scores are not zero.")
print("No definitive segment benchmark is computed until price quote comparability is resolved.")


## 18. Five record checks: expectations first
Expected: 2708 price 67.22 with missing bedroom retained; 2436587 price 72.30 with missing minimum stay retained; 6764023 value score 0 retained; 25870842 price 20583.33 retained; 607365253103904668 price 3.77 retained with missing bedroom and monthly quote flag. These fixed expectations come from the prior investigation and are declared before checking actual output.


In [ ]:
record_specs = [
    (2708, "price_numeric", 67.22, "Currency conversion"),
    (2436587, "price_numeric", 72.30, "Currency conversion; minimum_nights stays missing"),
    (6764023, "review_scores_value", 0.0, "Zero score retained"),
    (25870842, "price_numeric", 20583.33, "Extreme price retained"),
    (607365253103904668, "price_numeric", 3.77, "Low price retained; monthly quote flagged")
]
record_rows = []
for listing_id, field, expected, reason in record_specs:
    raw_record = listings.loc[listings["id"].eq(listing_id)].iloc[0]
    actual_record = listings_clean.loc[listings_clean["id"].eq(listing_id)].iloc[0]
    original = raw_record["price"] if field == "price_numeric" else raw_record[field]
    actual = actual_record[field]
    matched = abs(float(actual) - expected) < 0.000001
    extra_check = True
    if listing_id in [2708, 607365253103904668]:
        extra_check = pd.isna(actual_record["bedrooms"]) and not actual_record["eligible_price_bedroom_segment"]
    if listing_id == 2436587:
        extra_check = pd.isna(actual_record["minimum_nights"])
    if listing_id == 607365253103904668:
        extra_check = extra_check and bool(actual_record["monthly_quote_review_flag"])
    record_rows.append({"id": listing_id, "field": field, "original": original,
                        "expected": expected, "actual": actual, "reason": reason,
                        "matches": matched and extra_check})
record_checks = pd.DataFrame(record_rows)
display(record_checks)
assert record_checks["matches"].all()
print("Codex suggestion checked: price parsing has zero failures and an unchanged missing mask on the full dataset; fixed expected currency examples match.")


## 19. Final reconciliation and exports
Verify all original columns and row counts. Export the same 14 evidence/sample CSV files as the script. The diagnostic sample retains the five checked cases and difficult cases, followed by ordinary records to exactly 50 rows. No full tables are exported.


In [ ]:
final_rows = []
for name, raw in tables.items():
    cleaned = clean_tables[name]
    originals_unchanged = cleaned[raw.columns].equals(raw)
    added = [c for c in cleaned.columns if c not in raw.columns]
    assert len(cleaned) == len(raw)
    assert originals_unchanged
    assert added == approved_columns[name]
    final_rows.append({"table": name, "starting_rows": len(raw), "final_rows": len(cleaned),
                       "row_difference": len(cleaned) - len(raw),
                       "original_columns_unchanged": originals_unchanged,
                       "added_columns": ", ".join(added)})
final_reconciliation = pd.DataFrame(final_rows)
display(final_reconciliation)
sample_ids = [2708, 2436587, 6764023, 25870842, 607365253103904668]
for condition in [
    listings_clean["price_numeric"].isna(), listings_clean["bedrooms"].isna(),
    listings_clean["minimum_nights"].isna(), listings_clean["review_scores_value"].isna(),
    listings_clean["bedrooms"].eq(0), listings_clean["bedrooms"].ge(10),
    listings_clean["review_scores_value"].eq(0), listings_clean["minimum_nights"].ge(365)
]:
    for listing_id in listings_clean.loc[condition, "id"].head(3):
        if listing_id not in sample_ids:
            sample_ids.append(listing_id)
for listing_id in listings_clean["id"]:
    if len(sample_ids) >= 50:
        break
    if listing_id not in sample_ids:
        sample_ids.append(listing_id)
sample_columns = ["id", "neighbourhood_cleansed", "room_type", "property_type",
                  "bedrooms", "accommodates", "price", "price_numeric",
                  "minimum_nights", "review_scores_value", "number_of_reviews_ltm",
                  "availability_90", "last_scraped", "last_scraped_dt",
                  "has_observed_price", "has_bedroom_segment",
                  "eligible_price_bedroom_segment", "monthly_quote_review_flag"]
cleaned_sample = listings_clean.set_index("id").loc[sample_ids].reset_index()[sample_columns]
assert len(cleaned_sample) == 50
output_folder = data_folder.parent.parent / "cleaning_outputs"
output_folder.mkdir(exist_ok=True)
evidence_outputs = {
    "baseline": baseline, "baseline_missing": baseline_missing,
    "decision_log": decision_log, "price_evidence": price_evidence,
    "date_evidence": date_evidence, "missing_evidence": missing_evidence,
    "key_evidence": key_evidence, "range_evidence": range_evidence,
    "coverage_evidence": coverage_evidence, "text_evidence": text_evidence,
    "denominator_evidence": denominator_evidence, "record_checks": record_checks,
    "final_reconciliation": final_reconciliation, "cleaned_listings_sample_50": cleaned_sample
}
for name, evidence in evidence_outputs.items():
    evidence.to_csv(output_folder / (name + ".csv"), index=False)
print("Evidence output folder:", output_folder.resolve())
print("Exactly 50 diagnostic sample rows exported; no full tables exported.")


## 20. Reproducibility verification
Compare generated outputs against the previously verified script hashes and source files against their recorded hashes. This checks existing results without changing rules. The notebook is run from a restarted kernel, from original source files.


In [ ]:
import hashlib
import json

# Compare against the existing script verification evidence.
verification = json.loads(
    (output_folder / "reproducibility_checks.json").read_text(encoding="utf-8")
)
def file_hash(path):
    checksum = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            checksum.update(block)
    return checksum.hexdigest()

hash_rows = []
for filename, expected in verification["sha256"].items():
    actual = file_hash(output_folder / filename)
    hash_rows.append({"file": filename, "matches_script_output": actual == expected})
    assert actual == expected, filename
source_hash_rows = []
for filename, expected in verification["source_sha256"].items():
    actual = file_hash(data_folder / filename)
    source_hash_rows.append({"file": filename, "source_unchanged": actual == expected})
    assert actual == expected, filename

display(pd.DataFrame(hash_rows))
display(pd.DataFrame(source_hash_rows))
assert record_checks["matches"].all()
assert final_reconciliation["row_difference"].eq(0).all()
assert final_reconciliation["original_columns_unchanged"].all()
print("All 14 CSV output hashes match the script; all four source hashes are unchanged.")
print("All five record checks pass; every table retains its original rows and columns.")
